# SMILES-2026 LLM-Check Ablation

This notebook runs the expanded LLM-Check ablation over the paper-repo feature families:

- `logit`
- `hidden`
- `attns`
- concatenations of those families

Each feature set is evaluated with both logistic regression and a lightweight MLP to identify the best configuration.

In [ ]:
!git clone https://github.com/olgafilimonova2004/hallucination_detection_draft.git
%cd hallucination_detection_draft
!pip install -r requirements.txt

Optional: mount Google Drive so caches and ablation outputs persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/hallucination_detection/hallucination_detection_draft'
!mkdir -p {DRIVE_ROOT}/method3_llm_check/artifacts/cache
!mkdir -p {DRIVE_ROOT}/method3_llm_check/artifacts/ablation

Run the full LLM-Check feature-family ablation. The output JSON contains all tested feature sets, both classifiers, and the best configuration selected by mean validation accuracy.

In [ ]:
!python3 method3_llm_check/run_ablation.py \
  --data-file data/dataset.csv \
  --cache-file {DRIVE_ROOT}/method3_llm_check/artifacts/cache/method3_llm_check_cache.npz \
  --output-dir {DRIVE_ROOT}/method3_llm_check/artifacts/ablation \
  --feature-sets logit,hidden,attns,logit_hidden,logit_attns,hidden_attns,logit_hidden_attns \
  --batch-size 1 \
  --max-length 512 \
  --cache-dtype float32 \
  --hidden-dims 64,32 \
  --dropout-p 0.3 \
  --l2-weight-decay 1e-4

In [ ]:
import json
from pathlib import Path
import pandas as pd

results_path = Path(DRIVE_ROOT) / 'method3_llm_check/artifacts/ablation/ablation_results.json'
payload = json.loads(results_path.read_text())
leaderboard = pd.DataFrame(payload['configs'])
leaderboard[['name', 'feature_set', 'classifier', 'feature_dim', 'mean_val_accuracy', 'mean_test_accuracy']].head(20)

In [ ]:
payload['best_config']